## 1. Installation & Setup

In [1]:
import os
from getpass import getpass

# Set your Tinker API key
if "TINKER_API_KEY" not in os.environ:
    os.environ["TINKER_API_KEY"] = getpass("Enter your Tinker API key: ")

# (Optional) HuggingFace token if the dataset is private/gated
if "HF_TOKEN" not in os.environ:
    os.environ["HF_TOKEN"] = getpass("Enter your HuggingFace token (or press Enter to skip): ")

In [1]:
import tinker
from tinker import types
import numpy as np
import json
import time

# Verify connection to Tinker
service_client = tinker.ServiceClient()
capabilities = service_client.get_server_capabilities()
print("Connected to Tinker! Available models:")
for m in capabilities.supported_models:
    if "Qwen3" in m.model_name:
        print(f"  - {m.model_name}")

Connected to Tinker! Available models:
  - Qwen/Qwen3-235B-A22B-Instruct-2507
  - Qwen/Qwen3-30B-A3B
  - Qwen/Qwen3-30B-A3B-Base
  - Qwen/Qwen3-30B-A3B-Instruct-2507
  - Qwen/Qwen3-32B
  - Qwen/Qwen3-4B-Instruct-2507
  - Qwen/Qwen3-8B
  - Qwen/Qwen3-8B-Base
  - Qwen/Qwen3-VL-235B-A22B-Instruct
  - Qwen/Qwen3-VL-30B-A3B-Instruct
  - Qwen/Qwen3.5-27B
  - Qwen/Qwen3.5-35B-A3B
  - Qwen/Qwen3.5-397B-A17B
  - Qwen/Qwen3.5-397B-A17B:peft:262144
  - Qwen/Qwen3.5-4B
  - Qwen/Qwen3.6-27B
  - Qwen/Qwen3.6-35B-A3B


## 2. Load and Explore the Dataset

We load the `Alan123/homo-lumo-grpo` dataset from HuggingFace. This dataset contains chemistry problems related to HOMO-LUMO gap predictions.

> **Note:** Adjust the column names below (`"prompt"`, `"answer"`, etc.) to match the actual schema of your dataset. Run the exploration cell first to see the column names.

In [2]:
#配置代理环境

import os
import socket

# 1. 设置你探测到的 IP 或 OrbStack 专用域名
# 如果 ip route 返回的是 192.168.1.5，就填 http://192.168.1.5:7897
proxy_host = "172.18.48.1" # 如果是纯原生 Linux
# proxy_host = "host.orb.internal" # 如果是 macOS OrbStack
# proxy_host = "172.xx.xx.xx" # 如果是 WSL2/Docker，填 ip route 出来的网关

proxy_url = f"http://{proxy_host}:7897"

# 2. 先做最简单的“端口探活”，不用发 HTTP 请求
def test_port(ip, port):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.settimeout(2)
        return s.connect_ex((ip, port)) == 0

if test_port(proxy_host, 7897):
    print(f"✅ 成功发现代理服务：{proxy_url}")
    os.environ["http_proxy"] = proxy_url
    os.environ["https_proxy"] = proxy_url
    
    # 尝试加载数据
    from datasets import load_dataset
    try:
        # 这里建议先不传 token 试试，如果数据集是公开的
        dataset = load_dataset("Alan123/qa_dataset_with_qed_logp_mw_grpo")
        print("🚀 数据集加载成功！")
    except Exception as e:
        print(f"❌ 代理通了但下载失败: {e}")
else:
    print(f"❌ 依然无法连接到 {proxy_host}:7897。请确认 Clash 的 'Allow LAN' 已开启。")

✅ 成功发现代理服务：http://172.18.48.1:7897


🚀 数据集加载成功！


In [3]:
import os
import json
import time
from datasets import load_dataset, Dataset

# Keep original dataset directly. No four-step synthetic data generation.
print("Using original prompt/ground-truth dataset directly (no four-step synthesis).")

if "dataset" not in globals():
    dataset = load_dataset("Alan123/qa_dataset_with_qed_logp_mw_grpo")

if hasattr(dataset, "keys") and "train" in dataset:
    train_split = dataset["train"]
else:
    train_split = dataset

subset = train_split.select(range(min(10, len(train_split))))
print(f"Loaded train split size: {len(train_split)}")
print(f"Debug subset size: {len(subset)}")

Using original prompt/ground-truth dataset directly (no four-step synthesis).
Loaded train split size: 163415
Debug subset size: 10


In [4]:
# ============================================================================
# IMPORTANT: Update these column names based on your dataset's actual schema!
# ============================================================================
# After running the cell above, check the column names and update accordingly.
#
# Common patterns for GRPO chemistry datasets:
#   - "prompt" or "question" or "smiles" or "input" -> the question/molecule
#   - "answer" or "target" or "homo_lumo_gap" or "label" -> the ground truth
#   - "messages" -> pre-formatted chat messages
#
# We provide TWO paths below:
#   Path A: Dataset has "prompt" and "answer" columns
#   Path B: Dataset has pre-formatted "messages" column
#
# Uncomment the path that matches your dataset.

# --- Path A: prompt + answer columns ---
PROMPT_COL = "input"       # <-- UPDATE to your prompt column name
ANSWER_COL = "output"       # <-- UPDATE to your answer column name
DATA_SPLIT = "train"        # <-- UPDATE if needed

# --- Path B: pre-formatted messages ---
# MESSAGES_COL = "messages"  # <-- Uncomment if your dataset uses messages format

raw_data = dataset[DATA_SPLIT]
# raw_data = raw_data_origin[:10000]
print(raw_data)
print(f"Loaded {len(raw_data)} training examples")

Dataset({
    features: ['input', 'output'],
    num_rows: 163415
})
Loaded 163415 training examples


## 3. Configure the Model and Renderer

We'll use **Qwen3-8B** (a good balance of cost and capability). For larger-scale experiments, you can switch to `Qwen/Qwen3-30B-A3B` (MoE, very cost-effective) or `Qwen/Qwen3-32B` (dense).

Tinker uses a **Renderer** to convert chat messages into token sequences with proper special tokens and loss weights.

In [5]:
# Choose your base model
# Options: "Qwen/Qwen3-8B", "Qwen/Qwen3-30B-A3B", "Qwen/Qwen3-32B", etc.
BASE_MODEL = "Qwen/Qwen3-8B"

# LoRA rank: higher = more capacity but costs more. 32 is a good default.
LORA_RANK = 32

print(f"Base model: {BASE_MODEL}")
print(f"LoRA rank: {LORA_RANK}")

Base model: Qwen/Qwen3-8B
LoRA rank: 32


In [6]:
# Set up the renderer for Qwen3 chat format
# The renderer handles special tokens (<|im_start|>, <|im_end|>, etc.)
# and produces proper loss masks for training.

from tinker_cookbook import renderers, tokenizer_utils

tokenizer = tokenizer_utils.get_tokenizer(BASE_MODEL)
renderer = renderers.get_renderer('qwen3', tokenizer)

print(f"Tokenizer vocab size: {tokenizer.vocab_size}")
print(f"Stop sequences: {renderer.get_stop_sequences()}")

Tokenizer vocab size: 151643
Stop sequences: [151645]


## 4. Data Preparation Helpers

We need functions to convert our dataset into the format Tinker expects:
- **For SFT:** `types.Datum` with `model_input` and `loss_fn_inputs` (weights and target_tokens)
- **For GRPO:** Chat messages that can be rendered into prompts for sampling

In [7]:
# System prompt for chemistry optimization tasks
SYSTEM_PROMPT = (
    "You are a helpful chemistry assistant. "
    "You need to modify the input molecule to match specific property constraints."
)


def make_messages(example):
    """
    Convert a raw dataset example into a list of chat messages.
    Update this function based on your dataset's column schema.
    """
    # Path A: prompt + answer columns
    prompt_text = example[PROMPT_COL]
    answer_text = str(example[ANSWER_COL])

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": prompt_text},
        {"role": "assistant", "content": answer_text},
    ]
    return messages


    # Path B: pre-formatted messages
    # return example[MESSAGES_COL]


def make_prompt_messages(example, system_prompt=SYSTEM_PROMPT):
    """
    Create the prompt-only messages (no assistant response) for RL sampling.
    """
    prompt_text = example[PROMPT_COL]
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": prompt_text},
    ]
    return messages

In [8]:
def prepare_sft_datum(example):
    messages = make_messages(example)
    model_input, weights = renderer.build_supervised_example(messages)

    input_tokens = model_input.to_ints()
    weights_list = weights.tolist() if hasattr(weights, 'tolist') else list(weights)

    target_tokens = input_tokens[1:] + [input_tokens[-1]]
    weights_shifted = weights_list[1:] + [0]

    return types.Datum(
        model_input=types.ModelInput.from_ints(tokens=input_tokens[:-1]),
        loss_fn_inputs=dict(
            weights=weights_shifted[:-1],
            target_tokens=target_tokens[:-1],
        ),
    )



# Test: prepare one datum and inspect
test_datum = prepare_sft_datum(raw_data[0])
n_tokens = len(test_datum.model_input.to_ints())

# Debug: check what type the weights actually are
weights_val = test_datum.loss_fn_inputs['weights']
print(f"Weights type: {type(weights_val)}")
if hasattr(weights_val, '__len__') and len(weights_val) > 0:
    print(f"First element type: {type(weights_val[0])}, value: {weights_val[0]}")

Weights type: <class 'tinker.TensorData'>


## 5. Stage 1: Supervised Fine-Tuning (SFT)

SFT teaches the model the basic format and chemistry knowledge. We use the `cross_entropy` loss function, which trains the model to predict the next token for the assistant's response.

**Tinker's training loop:**
1. Prepare a batch of `Datum` objects
2. Call `forward_backward()` to compute gradients
3. Call `optim_step()` to update the LoRA weights
4. Repeat

In [9]:
SFT_LEARNING_RATE = 1e-4
SFT_BATCH_SIZE = 4           # Number of examples per gradient step
SFT_NUM_STEPS =  20          # Total training steps (adjust based on dataset size)
SFT_EVAL_EVERY = 1          # Evaluate every N steps
SFT_GRAD_ACCUM_STEPS = 1     # Gradient accumulation steps

In [ ]:


# Prepare all SFT data. Keep matching raw examples so format_r can be logged cheaply.
print("Preparing SFT training data...")
sft_data = []
sft_raw_examples = []
for i, example in enumerate(raw_data):
    try:
        datum = prepare_sft_datum(example)
        sft_data.append(datum)
        sft_raw_examples.append(example)
    except Exception as e:
        if i < 3:
            print(f"  Warning: skipping example {i}: {e}")

print(f"Prepared {len(sft_data)} SFT examples")

# Split into train/test
split_idx = int(0.9 * len(sft_data))
sft_train = sft_data[:split_idx]
sft_test = sft_data[split_idx:]
sft_train_examples = sft_raw_examples[:split_idx]
sft_test_examples = sft_raw_examples[split_idx:]
print(f"Train: {len(sft_train)}, Test: {len(sft_test)}")

Preparing SFT training data...


KeyboardInterrupt: 

In [13]:
sft_training_client = service_client.create_lora_training_client(
    base_model=BASE_MODEL,
    rank=LORA_RANK,
)
sft_checkpoint_path = "tinker://3b6c008f-c911-5983-8ca8-4b92099cd6f0:train:0/weights/Subprocess-sft500-fourstep-xml.Qwen3-8B" # <-- Set this to your checkpoint path if resuming
sft_training_client.load_state(sft_checkpoint_path)

In [18]:
# ========================
# SFT Training Loop
# ========================
import re

sft_metrics = []
rng = np.random.default_rng(42)

# Define format_r early so SFT can log target-output format before the RL cells.
if "format_r" not in globals():
    _sft_format_pat = re.compile(
        r"^(Replace|Add|Remove)\s+(.+?)(?:\s+with\s+(.+?))?\s+to\s+form\s+<SMILES>(.+)</SMILES>\.$",
        re.DOTALL,
    )

    def format_r(completions, **kwargs):
        return [1.0 if _sft_format_pat.fullmatch(str(text).strip()) else -4.0 for text in completions]

    print("Defined format_r for SFT logging with the strict action + <SMILES> pattern.")
else:
    print("Using existing format_r for SFT logging.")


def _call_format_r_batch(examples) -> tuple[float | None, float | None]:
    if not examples:
        return None, None
    outputs = [str(example[ANSWER_COL]) for example in examples]
    try:
        scores = format_r(outputs)
    except TypeError:
        scores = [format_r(text) for text in outputs]
    scores = np.array(scores, dtype=np.float64)
    return float(scores.mean()), float(np.mean(scores > 0.0))


def _fmt_metric(value, width: int = 8, digits: int = 3) -> str:
    if value is None:
        return f"{'-':>{width}}"
    return f"{value:>{width}.{digits}f}"


def _fmt_pct(value, width: int = 6) -> str:
    if value is None:
        return f"{'-':>{width}}"
    return f"{100.0 * value:>{width - 1}.1f}%"


print("Starting SFT training...")
header = (
    f"{'Step':>5} | {'Train Loss':>10} | {'Test Loss':>10} | "
    f"{'TrainFmt':>8} | {'TrFmt%':>6} | {'TestFmt':>8} | {'TeFmt%':>6} | {'Time (s)':>8}"
)
print(header)
print("-" * len(header))

for step in range(SFT_NUM_STEPS):
    t0 = time.time()

    # Sample a mini-batch
    batch_indices = rng.choice(len(sft_train), size=SFT_BATCH_SIZE, replace=False)
    batch = [sft_train[i] for i in batch_indices]
    batch_examples = [sft_train_examples[i] for i in batch_indices]

    # Forward-backward pass with cross-entropy loss
    fwdbwd_future = sft_training_client.forward_backward(batch, loss_fn="cross_entropy")

    # Optimizer step
    optim_future = sft_training_client.optim_step(
        types.AdamParams(learning_rate=SFT_LEARNING_RATE)
    )

    # Wait for results
    fwdbwd_result = fwdbwd_future.result()
    optim_result = optim_future.result()

    # Compute train loss
    logprobs = np.concatenate(
        [out["logprobs"].tolist() for out in fwdbwd_result.loss_fn_outputs]
    )
    weights = np.concatenate(
        [d.loss_fn_inputs["weights"].tolist() if hasattr(d.loss_fn_inputs["weights"], "tolist")
         else d.loss_fn_inputs["weights"]
         for d in batch]
    )
    weights_arr = np.array(weights, dtype=float)
    train_loss = -np.dot(logprobs, weights_arr) / max(weights_arr.sum(), 1)
    train_format_r, train_format_pass_rate = _call_format_r_batch(batch_examples)

    elapsed = time.time() - t0
    metrics_entry = {
        "step": step,
        "train_loss": float(train_loss),
        "train_format_r": train_format_r,
        "train_format_pass_rate": train_format_pass_rate,
        "time": elapsed,
    }

    # Periodic evaluation on test set. Format is checked on target outputs, with no sampling cost.
    if step % SFT_EVAL_EVERY == 0 and len(sft_test) > 0:
        test_batch = sft_test[:min(SFT_BATCH_SIZE, len(sft_test))]
        test_batch_examples = sft_test_examples[:min(SFT_BATCH_SIZE, len(sft_test_examples))]
        test_fwdbwd = sft_training_client.forward_backward(test_batch, loss_fn="cross_entropy")
        # Note: we do not call optim_step for test — just forward pass.
        test_result = test_fwdbwd.result()
        test_logprobs = np.concatenate(
            [out["logprobs"].tolist() for out in test_result.loss_fn_outputs]
        )
        test_weights = np.concatenate(
            [d.loss_fn_inputs["weights"].tolist() if hasattr(d.loss_fn_inputs["weights"], "tolist")
             else d.loss_fn_inputs["weights"]
             for d in test_batch]
        )
        test_weights_arr = np.array(test_weights, dtype=float)
        test_loss = -np.dot(test_logprobs, test_weights_arr) / max(test_weights_arr.sum(), 1)
        test_format_r, test_format_pass_rate = _call_format_r_batch(test_batch_examples)

        metrics_entry["test_loss"] = float(test_loss)
        metrics_entry["test_format_r"] = test_format_r
        metrics_entry["test_format_pass_rate"] = test_format_pass_rate

        print(
            f"{step:>5} | {train_loss:>10.4f} | {test_loss:>10.4f} | "
            f"{_fmt_metric(train_format_r)} | {_fmt_pct(train_format_pass_rate)} | "
            f"{_fmt_metric(test_format_r)} | {_fmt_pct(test_format_pass_rate)} | {elapsed:>8.1f}"
        )
    else:
        print(
            f"{step:>5} | {train_loss:>10.4f} | {'-':>10} | "
            f"{_fmt_metric(train_format_r)} | {_fmt_pct(train_format_pass_rate)} | "
            f"{'-':>8} | {'-':>6} | {elapsed:>8.1f}"
        )

    sft_metrics.append(metrics_entry)

print("\nSFT training complete!")

Using existing format_r for SFT logging.
Starting SFT training...
 Step | Train Loss |  Test Loss | TrainFmt | TrFmt% |  TestFmt | TeFmt% | Time (s)
----------------------------------------------------------------------------------
    0 |     0.1653 |     0.1105 |    1.000 | 100.0% |    1.000 | 100.0% |      3.0
    1 |     0.2015 |     0.0795 |    1.000 | 100.0% |    1.000 | 100.0% |      3.2
    2 |     0.1855 |     0.0575 |    1.000 | 100.0% |    1.000 | 100.0% |      4.0
    3 |     0.1446 |     0.0416 |    1.000 | 100.0% |    1.000 | 100.0% |      2.3
    4 |     0.1596 |     0.0322 |    1.000 | 100.0% |    1.000 | 100.0% |      3.2
    5 |     0.4147 |     0.0259 |    1.000 | 100.0% |    1.000 | 100.0% |      3.2
    6 |     0.5194 |     0.0237 |    1.000 | 100.0% |    1.000 | 100.0% |      3.4
    7 |     0.3902 |     0.0227 |    1.000 | 100.0% |    1.000 | 100.0% |      4.3
    8 |     0.4660 |     0.0219 |    1.000 | 100.0% |    1.000 | 100.0% |      2.5
    9 |     0.2938 | 

In [ ]:
# Save the SFT checkpoint
# This saves the LoRA weights + optimizer state so we can:
#   1. Sample from the SFT model
#   2. Continue training with GRPO

sft_save_result = sft_training_client.save_state(name="").result()
sft_checkpoint_path = sft_save_result.path
print(f"SFT checkpoint saved at: {sft_checkpoint_path}")


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(
    [m["step"] for m in sft_metrics],
    [m["train_loss"] for m in sft_metrics],
    label="Train Loss",
    alpha=0.7,
)
test_metrics = [m for m in sft_metrics if "test_loss" in m]
if test_metrics:
    axes[0].plot(
        [m["step"] for m in test_metrics],
        [m["test_loss"] for m in test_metrics],
        "o-",
        label="Test Loss",
    )
axes[0].set_xlabel("Step")
axes[0].set_ylabel("Loss (NLL per token)")
axes[0].set_title("SFT Loss")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

train_format_metrics = [m for m in sft_metrics if "train_format_r" in m]
test_format_metrics = [m for m in sft_metrics if "test_format_r" in m]
if train_format_metrics:
    axes[1].plot(
        [m["step"] for m in train_format_metrics],
        [m["train_format_r"] for m in train_format_metrics],
        ".-",
        label="train format_r",
    )
    axes[1].plot(
        [m["step"] for m in train_format_metrics],
        [m["train_format_pass_rate"] for m in train_format_metrics],
        ".-",
        label="train format pass rate",
    )
if test_format_metrics:
    axes[1].plot(
        [m["step"] for m in test_format_metrics],
        [m["test_format_r"] for m in test_format_metrics],
        "o-",
        label="test format_r",
    )
    axes[1].plot(
        [m["step"] for m in test_format_metrics],
        [m["test_format_pass_rate"] for m in test_format_metrics],
        "s-",
        label="test format pass rate",
    )
if train_format_metrics or test_format_metrics:
    axes[1].set_title("SFT Target Format")
    axes[1].set_ylabel("Score / Rate")
    axes[1].legend()
else:
    axes[1].set_title("SFT Target Format")
    axes[1].text(0.5, 0.5, "No format metrics logged", ha="center", va="center")
axes[1].set_xlabel("Step")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---

## 7. Stage 2: Reinforcement Learning

Now we switch to **GRPO-style RL training** to improve the model's accuracy using a reward signal.

**How GRPO works with Tinker:**
1. For each prompt, **sample G completions** from the current model
2. **Score** each completion with a reward function
3. Compute **group-normalized advantages** (subtract mean, divide by std within each group)
4. Use the `ppo` (or `importance_sampling`) loss to update the policy

Tinker's built-in `ppo` loss handles the clipped surrogate objective. The GRPO-specific part (group normalization of advantages) is done on the client side.

---

## 关于qed logp mw的rewardFunc

In [10]:
import re
from collections import Counter
from typing import Dict, List, Optional, Tuple

import numpy as np
from rdkit import Chem, DataStructs, RDLogger
from rdkit.Chem import QED, Descriptors, Crippen, rdFingerprintGenerator

RDLogger.DisableLog("rdApp.*")

FORMAT_PENALTY = -10.0
INVALIDITY_PENALTY = -10.0
REPETITION_PENALTY_STEP = 0.1

_FORMAT_RE = re.compile(
    r"^(Replace|Add|Remove)\s+(.+?)(?:\s+with\s+(.+?))?\s+to\s+form\s+<SMILES>(.+)</SMILES>\.$",
    re.DOTALL,
)


def _normalize_completion_text(text: str | None) -> str:
    """Drop think blocks and surrounding spaces."""
    if not text:
        return ""
    clean = re.sub(r"<think>.*?</think>", "", str(text), flags=re.DOTALL)
    return clean.replace("<|im_end|>", "").strip()


def extract_smiles(text: str | None) -> str | None:
    """Extract the first SMILES inside <SMILES>...</SMILES>."""
    if not text:
        return None
    m = re.search(r"<SMILES>(.*?)</SMILES>", str(text), flags=re.DOTALL)
    return m.group(1).strip() if m else None


def strict_format_reward(completions: List[str], **kwargs) -> List[float]:
    """Format-only reward: valid action sentence gets +1, malformed output gets -10."""
    rewards = []
    for text in completions:
        normalized = _normalize_completion_text(text)
        rewards.append(1.0 if _FORMAT_RE.fullmatch(normalized) else FORMAT_PENALTY)
    return rewards


def format_r(completions, **kwargs):
    """Compatibility alias used by earlier SFT logging cells."""
    if isinstance(completions, str):
        return strict_format_reward([completions], **kwargs)[0]
    return strict_format_reward(list(completions), **kwargs)


def _property_values(mol) -> dict[str, float]:
    return {
        "qed": float(QED.qed(mol)),
        "logp": float(Crippen.MolLogP(mol)),
        "mw": float(Descriptors.ExactMolWt(mol)),
    }


def _parse_property_targets(prompt_text: str, mol_in):
    """Return absolute QED/LogP/MW targets derived from the current prompt."""
    specs = {}
    for prop in ("QED", "LogP", "MW"):
        m = re.search(rf"<{prop}>(-?\d*\.?\d+)</{prop}>\s*(higher|lower)", prompt_text)
        if not m:
            return None
        delta = float(m.group(1))
        direction = 1.0 if m.group(2).lower() == "higher" else -1.0
        specs[prop.lower()] = (delta, direction)

    input_props = _property_values(mol_in)
    target = {
        "qed": max(0.01, min(0.99, input_props["qed"] + specs["qed"][1] * specs["qed"][0])),
        "logp": input_props["logp"] + specs["logp"][1] * specs["logp"][0],
        "mw": input_props["mw"] + specs["mw"][1] * specs["mw"][0],
    }
    return target, input_props


def property_error(mol, target: dict[str, float]) -> tuple[float, dict[str, float], dict[str, float]]:
    """Normalized MAE against absolute targets."""
    props = _property_values(mol)
    parts = {
        "qed_err": abs(props["qed"] - target["qed"]) / 0.5,
        "logp_err": abs(props["logp"] - target["logp"]) / 3.0,
        "mw_err": abs(props["mw"] - target["mw"]) / 50.0,
    }
    total = float(parts["qed_err"] + parts["logp_err"] + parts["mw_err"])
    return total, parts, props


def normalized_total_error(qed_error: float, logp_error: float, mw_error: float) -> float:
    """NTE/enorm = |dQED| + |dLogP|/10 + |dMW|/700."""
    return float(abs(qed_error) + abs(logp_error) / 10.0 + abs(mw_error) / 700.0)


def compute_final_nte_details(final_smiles: str | None, initial_prompt_text: str) -> dict:
    """Evaluate final generated molecule against the original prompt target."""
    details = {
        "valid": False,
        "failure": None,
        "nte": None,
        "qed_error": None,
        "logp_error": None,
        "mw_error": None,
        "qed_target": None,
        "logp_target": None,
        "mw_target": None,
        "qed_out": None,
        "logp_out": None,
        "mw_out": None,
    }
    in_smiles = extract_smiles(initial_prompt_text)
    if not final_smiles:
        details["failure"] = "missing_final_smiles"
        return details
    if not in_smiles:
        details["failure"] = "missing_initial_smiles"
        return details

    mol_in = Chem.MolFromSmiles(in_smiles)
    mol_out = Chem.MolFromSmiles(final_smiles)
    if mol_in is None:
        details["failure"] = "invalid_initial_smiles"
        return details
    if mol_out is None:
        details["failure"] = "invalid_final_smiles"
        return details

    parsed = _parse_property_targets(initial_prompt_text, mol_in)
    if parsed is None:
        details["failure"] = "target_parse_failed"
        return details
    target, _input_props = parsed
    out_props = _property_values(mol_out)

    qed_error = abs(out_props["qed"] - target["qed"])
    logp_error = abs(out_props["logp"] - target["logp"])
    mw_error = abs(out_props["mw"] - target["mw"])
    details.update(
        {
            "valid": True,
            "failure": None,
            "nte": normalized_total_error(qed_error, logp_error, mw_error),
            "qed_error": float(qed_error),
            "logp_error": float(logp_error),
            "mw_error": float(mw_error),
            "qed_target": float(target["qed"]),
            "logp_target": float(target["logp"]),
            "mw_target": float(target["mw"]),
            "qed_out": float(out_props["qed"]),
            "logp_out": float(out_props["logp"]),
            "mw_out": float(out_props["mw"]),
        }
    )
    return details


print("Single-stage reward utilities loaded: format-only check, MAE property reward, final NTE helpers.")


Single-stage reward utilities loaded: format-only check, MAE property reward, final NTE helpers.


In [11]:
"""Single-stage GRPO reward: R = format_r + prop_r + repetition_penalty."""


def compute_property_reward_details(completion_text: str, prompt_text: str) -> dict:
    text = _normalize_completion_text(completion_text)
    in_smiles = extract_smiles(prompt_text)
    out_smiles = extract_smiles(text)
    details = {
        "property_reward": float(INVALIDITY_PENALTY),
        "property_failure": None,
        "total_err": None,
        "qed_err": None,
        "logp_err": None,
        "mw_err": None,
        "qed_target": None,
        "logp_target": None,
        "mw_target": None,
        "qed_out": None,
        "logp_out": None,
        "mw_out": None,
    }
    if not in_smiles or not out_smiles:
        details["property_failure"] = "missing_smiles"
        return details

    mol_in = Chem.MolFromSmiles(in_smiles)
    mol_out = Chem.MolFromSmiles(out_smiles)
    if mol_in is None or mol_out is None:
        details["property_failure"] = "invalid_smiles"
        return details

    parsed = _parse_property_targets(prompt_text, mol_in)
    if parsed is None:
        details["property_failure"] = "target_parse_failed"
        return details
    target, _input_props = parsed

    total_err, parts, output_props = property_error(mol_out, target)
    property_reward = float(1.0 - total_err)
    details.update(
        {
            "property_reward": property_reward,
            "property_failure": None,
            "total_err": float(total_err),
            "qed_err": float(parts["qed_err"]),
            "logp_err": float(parts["logp_err"]),
            "mw_err": float(parts["mw_err"]),
            "qed_target": float(target["qed"]),
            "logp_target": float(target["logp"]),
            "mw_target": float(target["mw"]),
            "qed_out": float(output_props["qed"]),
            "logp_out": float(output_props["logp"]),
            "mw_out": float(output_props["mw"]),
        }
    )
    return details


def prop_r(completions: List[str], **kwargs) -> List[float]:
    prompts: List[str] = kwargs["prompts"]
    return [
        float(compute_property_reward_details(comp, prompt)["property_reward"])
        for comp, prompt in zip(completions, prompts)
    ]


# Compatibility name used by earlier notebook cells.
def multi_property_reward(completions: List[str], **kwargs) -> List[float]:
    return prop_r(completions, **kwargs)


def compute_reward_components(
    completion_text: str,
    prompt_text: str,
    ground_truth: str | None = None,
    *,
    repetition_penalty: float = 0.0,
    repetition_count: int = 0,
    **_unused,
) -> dict:
    text = _normalize_completion_text(completion_text)
    pred_smiles = extract_smiles(text)
    mol_pred = Chem.MolFromSmiles(pred_smiles) if pred_smiles else None

    format_score = float(strict_format_reward([text])[0])
    prop_details = compute_property_reward_details(text, prompt_text)
    property_score = float(prop_details["property_reward"])
    repetition_score = float(repetition_penalty)
    total = float(format_score + property_score + repetition_score)

    out = {
        "strict": format_score,
        "format": format_score,
        "property": property_score,
        "repetition": repetition_score,
        "repetition_count": int(repetition_count),
        "total": total,
        "has_smiles": bool(pred_smiles),
        "valid_smiles": mol_pred is not None,
        "valid_format": format_score > 0.0,
    }
    out.update(prop_details)
    return out


def compute_reward(
    completion_text: str,
    prompt_text: str,
    ground_truth: str | None = None,
    **kwargs,
) -> float:
    return float(
        compute_reward_components(
            completion_text,
            prompt_text,
            ground_truth,
            **kwargs,
        )["total"]
    )


print("Single-stage MAE reward active: total = format + property + prompt-group repetition penalty.")


Single-stage MAE reward active: total = format + property + prompt-group repetition penalty.


# Baseline v2

In [12]:
# Baseline-v2 reward override disabled.
# The active GRPO reward is defined above as:
#   total = format_r + prop_r + repetition_penalty
# with no reward weights, no similarity term, no direction bonus, and no format gate.
print("Baseline-v2 reward override disabled; using single-stage MAE reward.")


Baseline-v2 reward override disabled; using single-stage MAE reward.


In [13]:
# Group-normalized advantages used by GRPO.
def compute_group_advantages(rewards: list[float]) -> list[float]:
    """
    GRPO-style group normalization: (r - mean) / std.
    Return zeros when std is near zero to avoid NaN.
    """
    arr = np.array(rewards, dtype=np.float64)
    std = arr.std()
    if std < 1e-8:
        return [0.0] * len(rewards)
    return ((arr - arr.mean()) / std).tolist()

In [14]:
# Shared helpers for multi-turn RL.

import re
import numpy as np
from typing import Optional

import tinker
from tinker import types
from rdkit import Chem, RDLogger
from rdkit.Chem import QED, Descriptors, Crippen

# Per-property tolerance for deciding whether a target is truly met.
TARGET_TOLERANCE = {
    "QED": 0.02,
    "LogP": 0.20,
    "MW": 5.0,
}


def _mol_from_prompt(prompt_text: str) -> Optional[Chem.Mol]:
    """Extract the input molecule from prompt <SMILES> tag."""
    m = re.search(r"<SMILES>(.*?)</SMILES>", prompt_text)
    if not m:
        return None
    return Chem.MolFromSmiles(m.group(1))


def _target_progress_lines(mol_in: Chem.Mol, mol_out: Chem.Mol, prompt_text: str) -> tuple[list[str], list[str]]:
    """
    Build per-property target progress lines.

    Returns:
      misses: properties that have not met their target by tolerance
      lines: concise human-readable lines with base/current/target and gap
    """
    prop_specs = [
        ("QED", QED.qed, "{:.3f}"),
        ("LogP", Crippen.MolLogP, "{:.3f}"),
        ("MW", Descriptors.ExactMolWt, "{:.3f}"),
    ]

    misses = []
    lines = []

    for prop, fn, fmt in prop_specs:
        m = re.search(rf"<{prop}>(-?\d*\.?\d+)</{prop}>\s*(higher|lower)", prompt_text)
        if not m:
            continue

        delta = float(m.group(1))
        direction = m.group(2).lower()

        base_val = float(fn(mol_in))
        current_val = float(fn(mol_out))
        sign = 1.0 if direction == "higher" else -1.0
        target_val = base_val + sign * delta

        if prop == "QED":
            target_val = max(0.01, min(0.99, target_val))

        # signed_need > 0: still short in required direction; <= 0: reached/passed direction.
        if direction == "higher":
            signed_need = target_val - current_val
            action = "increase"
        else:
            signed_need = current_val - target_val
            action = "decrease"

        still_needed = abs(signed_need)
        tol = TARGET_TOLERANCE.get(prop, 0.0)
        # direction_met means whether current change follows required direction from base.
        # It is independent from whether target tolerance has been reached.
        eps = 1e-12
        if direction == "higher":
            direction_met = (current_val - base_val) >= -eps
        else:
            direction_met = (base_val - current_val) >= -eps
        met = still_needed <= tol

        if signed_need > 0:
            progress_desc = f"need {fmt.format(still_needed)} more {action}"
        elif signed_need < 0:
            progress_desc = f"overshot by {fmt.format(still_needed)} in {action} direction"
        else:
            progress_desc = "exactly on target"

        if not met:
            misses.append(prop)

        lines.append(
            f"{prop}: base={fmt.format(base_val)}, current={fmt.format(current_val)}, "
            f"target={fmt.format(target_val)} ({direction} by {delta:.3f}), "
            f"still_needed={fmt.format(still_needed)} ({progress_desc}), "
            f"tol={fmt.format(tol)}, direction_met={direction_met}, met={met}"
        )

    return misses, lines


def _run_env(completion_text: str, prompt_text: str) -> tuple[str, Optional[Chem.Mol], bool]:
    """
    Environment step for multi-turn RL.

    prompt_text should be the CURRENT turn prompt, so base values are computed
    against the current molecule rather than always the initial molecule.
    """
    text = _normalize_completion_text(completion_text)
    pred_smiles = extract_smiles(text)
    if not pred_smiles:
        return "Your output did not contain a valid <SMILES>...</SMILES> block. Please revise.", None, False

    mol = Chem.MolFromSmiles(pred_smiles)
    if mol is None:
        return "The SMILES you provided is chemically invalid. Please revise.", None, False

    qed_val = QED.qed(mol)
    logp_val = Crippen.MolLogP(mol)
    mw_val = Descriptors.ExactMolWt(mol)

    mol_in = _mol_from_prompt(prompt_text)
    if mol_in is None:
        feedback = (
            f"Environment feedback: QED={qed_val:.3f}, LogP={logp_val:.2f}, MW={mw_val:.1f}. "
            "Input molecule is missing/invalid in prompt, so target gap is unavailable."
        )
        return feedback, mol, False

    misses, progress_lines = _target_progress_lines(mol_in, mol, prompt_text)
    all_met = len(progress_lines) > 0 and len(misses) == 0

    if progress_lines:
        progress_block = "\n- " + "\n- ".join(progress_lines)
    else:
        progress_block = "\n- No parseable target tags (<QED>/<LogP>/<MW>) found in prompt."

    if all_met:
        summary = "All targets met within tolerance. Keep output concise and chemically valid."
    else:
        unmet = ", ".join(misses) if misses else "unknown"
        summary = f"Targets not met (by target gap tolerance): {unmet}. Please refine toward target values."

    feedback = (
        f"Environment feedback: QED={qed_val:.3f}, LogP={logp_val:.2f}, MW={mw_val:.1f}."
        f"\nTarget progress:{progress_block}"
        f"\n{summary}"
    )
    return feedback, mol, all_met

In [15]:
# RL prompt: single action output only (no four-step chain-of-thought).
SYSTEM_PROMPT_RL = """
You are a helpful chemistry assistant for multi-turn molecule optimization.

Rules:
1) Output exactly one edit action sentence in one of these formats:
   - Replace <old_fragment> with <new_fragment> to form <SMILES>new_smiles_here</SMILES>.
   - Add <fragment> to form <SMILES>new_smiles_here</SMILES>.
   - Remove <fragment> to form <SMILES>new_smiles_here</SMILES>.
2) Do not output analysis/plan/verification/think blocks.
3) In turn >= 2, if the user provides a 'Current candidate molecule' SMILES,
   you must edit that candidate molecule, not the original input molecule.
"""

In [16]:
# ══════════════════════════════════════════════════════════════════════════════
#  修复5：Mode B 的组归一化 + EMA baseline
#
#  设计思路：
#    两个机制各司其职，解决不同层面的问题：
#
#    ① 组归一化（intra-prompt）
#       对同一 prompt 采 K 条轨迹，计算每条轨迹的折扣回报 G_k，
#       再做 (G_k - mean) / std 归一化。
#       作用：告诉模型"在同一起点下，哪种推理路径更优"。
#       这是 GRPO 的核心信号，Mode B 必须保留。
#
#    ② EMA baseline（inter-prompt）
#       维护一个跨 prompt、跨 iteration 的滑动平均 baseline。
#       在组归一化之后再减去 baseline（可选，作为二次校正）。
#       作用：防止某些 prompt 天然容易（reward 整体高）导致
#       advantage 尺度膨胀，稳定不同 prompt 之间的梯度大小。
#
#    两者的关系：
#      组归一化是"排名"，EMA 是"定零点"。
#      单独用 EMA（原始建议）：有零点但没有排名 → 信号弱。
#      只用组归一化：排名准确但跨 prompt 尺度不齐 → 梯度方差大。
#      两者结合：既有排名又有稳定尺度 → 最优。
#
#    关于 K 的选择：
#      Mode A：K=4~8（每条只需 1 次采样）
#      Mode B：K=2~3（每条需要 max_turns 次采样，成本是 Mode A 的 max_turns 倍）
#      K=1 时退化为普通 PG，组归一化失效，不建议使用。
# ══════════════════════════════════════════════════════════════════════════════
 
class EMABaseline:
    """
    指数移动平均 baseline，用于跨 prompt 稳定 advantage 的绝对尺度。
 
    update(G) 先返回当前 baseline 再更新，使得当前 batch 使用
    "历史经验"而非"本轮数据"作为参考，避免自举偏差。
    """
    def __init__(self, alpha: float = 0.05, init: float = 0.0):
        self.alpha   = alpha
        self.value   = init
        self._warmed = False  # 前几个样本用来热身，不参与训练
 
    def update_and_get(self, G: float) -> float:
        """返回当前 baseline，然后更新。"""
        current = self.value
        if not self._warmed:
            # 第一个样本直接初始化，避免初始 baseline=0 造成偏差
            self.value   = G
            self._warmed = True
        else:
            self.value = (1 - self.alpha) * self.value + self.alpha * G
        return current

In [17]:
# Multi-turn RL with environment feedback, rollback, and prompt-group repetition penalty.
def run_mode_b(
    example: dict,
    renderer,
    sampling_client,
    tokenizer,
    ema_baseline: EMABaseline,
    num_trajectories: int = 8,
    max_turns: int = 3,
    gamma: float = 0.85,
    max_tokens: int = 256,
    temperature: float = 1.0,
    top_p: float = 0.9,
    top_k: int = 50,
    debug: bool = False,
    debug_prefix: str = "",
) -> tuple[list[types.Datum], list[float], dict]:
    """
    Multi-turn GRPO rollout.

    - Each trajectory keeps an independent current molecule state.
    - Invalid molecules are scored, but the next turn rolls back to the last valid molecule.
    - Repetition penalty is computed only within this prompt's trajectory group.
    """
    prompt_text = str(example[PROMPT_COL])
    ground_truth = str(example[ANSWER_COL])

    def _replace_prompt_smiles(base_prompt: str, current_smiles: str) -> str:
        """Replace current molecule and refresh target deltas while preserving absolute targets."""
        new_prompt = re.sub(
            r"<SMILES>.*?</SMILES>",
            lambda _m: f"<SMILES>{current_smiles}</SMILES>",
            base_prompt,
            count=1,
            flags=re.DOTALL,
        )

        fragment_fn = globals().get("get_clean_fragments")
        if callable(fragment_fn):
            try:
                fragment_list = fragment_fn(current_smiles)
            except Exception:
                fragment_list = []
        else:
            fragment_list = []
        if not fragment_list:
            fragment_list = [current_smiles]

        fragment_repr = repr(fragment_list)
        new_prompt = re.sub(
            r"(<SMILES>.*?</SMILES>)(?:,\s*which is composed of fragments\s+.*?\.)?",
            lambda m: f"{m.group(1)}, which is composed of fragments {fragment_repr}.",
            new_prompt,
            count=1,
            flags=re.DOTALL,
        )

        mol_old = _mol_from_prompt(base_prompt)
        mol_new = Chem.MolFromSmiles(current_smiles)
        if mol_old is None or mol_new is None:
            return new_prompt

        prop_specs = [
            ("QED", QED.qed, True),
            ("LogP", Crippen.MolLogP, False),
            ("MW", Descriptors.ExactMolWt, False),
        ]
        for prop, fn, is_qed in prop_specs:
            m = re.search(rf"<{prop}>(-?\d*\.?\d+)</{prop}>\s*(higher|lower)", new_prompt)
            if not m:
                continue
            old_delta = float(m.group(1))
            old_dir = m.group(2).lower()
            old_base = float(fn(mol_old))
            target_abs = old_base + (old_delta if old_dir == "higher" else -old_delta)
            if is_qed:
                target_abs = max(0.01, min(0.99, target_abs))

            new_base = float(fn(mol_new))
            new_dir = "higher" if target_abs >= new_base else "lower"
            new_delta = abs(target_abs - new_base)
            new_prompt = re.sub(
                rf"<{prop}>-?\d*\.?\d+</{prop}>\s*(higher|lower)",
                f"<{prop}>{new_delta:.3f}</{prop}> {new_dir}",
                new_prompt,
                count=1,
            )
        return new_prompt

    def _compact_env_feedback(feedback_text: str) -> str:
        """Keep only headline + final summary to avoid repeating target lines."""
        if not feedback_text:
            return ""
        lines = [ln.strip() for ln in feedback_text.splitlines() if ln.strip()]
        if not lines:
            return feedback_text
        if lines[0].startswith("Environment feedback:"):
            if len(lines) == 1:
                return lines[0]
            tail = lines[-1]
            if tail == lines[0]:
                return lines[0]
            return lines[0] + "\n" + tail
        return feedback_text

    def _rollback_state_feedback(current_prompt: str) -> tuple[str, str, bool]:
        rollback_mol = _mol_from_prompt(current_prompt)
        if rollback_mol is None:
            feedback = "Environment feedback: rollback molecule is missing or invalid in the current prompt."
            guidance = (
                "all_met=False; unmet_properties=unknown.\n"
                "- target-gap details unavailable because the rollback molecule is invalid."
            )
            return feedback, guidance, False

        qed_val = QED.qed(rollback_mol)
        logp_val = Crippen.MolLogP(rollback_mol)
        mw_val = Descriptors.ExactMolWt(rollback_mol)
        misses, progress_lines = _target_progress_lines(rollback_mol, rollback_mol, current_prompt)
        all_met_state = bool(progress_lines) and len(misses) == 0
        unmet_text = ", ".join(misses) if misses else ("none" if progress_lines else "unknown")
        guidance_block = (
            "\n".join([f"- {line}" for line in progress_lines])
            if progress_lines
            else "- target-gap details unavailable."
        )
        progress_block = (
            "\n- " + "\n- ".join(progress_lines)
            if progress_lines
            else "\n- No parseable target tags (<QED>/<LogP>/<MW>) found in prompt."
        )
        if all_met_state:
            summary = "Rollback molecule already meets all targets within tolerance."
        else:
            summary = f"Rollback molecule targets not met: {unmet_text}. Continue refining from this valid molecule."
        feedback = (
            "Environment feedback: previous attempt invalid/missing; using previous valid molecule "
            f"with QED={qed_val:.3f}, LogP={logp_val:.2f}, MW={mw_val:.1f}."
            f"\nTarget progress:{progress_block}"
            f"\n{summary}"
        )
        guidance = f"all_met={all_met_state}; unmet_properties={unmet_text}.\n{guidance_block}"
        return feedback, guidance, all_met_state

    initial_mol = _mol_from_prompt(prompt_text)
    initial_smiles = extract_smiles(prompt_text)
    initial_canonical = Chem.MolToSmiles(initial_mol) if initial_mol is not None else None

    K = int(num_trajectories)
    traj_states = [
        {
            "turns": [],
            "current_prompt": prompt_text,
            "prev_env_feedback": "",
            "prev_guidance": "",
            "done": False,
            "current_valid_smiles": initial_smiles if initial_canonical else None,
            "current_valid_canonical": initial_canonical,
            "has_valid_generated": False,
            "last_valid_generated_smiles": None,
            "last_valid_generated_canonical": None,
        }
        for _ in range(K)
    ]

    for turn_idx in range(max_turns):
        active_indices = [k for k in range(K) if not traj_states[k]["done"]]
        if not active_indices:
            break

        prompt_inputs = {}
        prompt_tokens = {}
        for k in active_indices:
            st = traj_states[k]
            messages = [
                {"role": "system", "content": SYSTEM_PROMPT_RL},
                {"role": "user", "content": st["current_prompt"]},
            ]
            if turn_idx > 0 and (st["prev_env_feedback"] or st["prev_guidance"]):
                feedback_parts = [
                    "Environment feedback from your previous attempt:",
                    st["prev_env_feedback"],
                    "",
                    "Target-gap priorities from previous turn (must address first):",
                    st["prev_guidance"],
                    "",
                    "Interpretation rules (must follow):",
                    "- If a property says 'need ... more increase/decrease', move in that direction.",
                    "- If a property says 'overshot by ...', reverse slightly toward target.",
                    "- Keep already-met properties near target while fixing unmet ones.",
                    "",
                    "Use the current molecule shown in the prompt above as the only edit base.",
                    "Do not restart from the original molecule.",
                ]
                messages.append({"role": "user", "content": "\n".join(feedback_parts).strip()})

            pi = renderer.build_generation_prompt(messages)
            prompt_inputs[k] = pi
            prompt_tokens[k] = pi.to_ints()

        futures = {
            k: sampling_client.sample(
                prompt=prompt_inputs[k],
                sampling_params=types.SamplingParams(
                    max_tokens=max_tokens,
                    temperature=temperature,
                    top_p=top_p,
                    top_k=top_k,
                    stop=renderer.get_stop_sequences(),
                ),
                num_samples=1,
            )
            for k in active_indices
        }

        for k in active_indices:
            result = futures[k].result()
            seq = result.sequences[0]
            raw_completion_text = tokenizer.decode(seq.tokens, skip_special_tokens=True)
            completion_text = _normalize_completion_text(raw_completion_text)

            st = traj_states[k]
            reward_components = compute_reward_components(
                completion_text,
                st["current_prompt"],
                ground_truth,
            )
            turn_reward = float(reward_components["total"])
            env_feedback, mol_out, all_met = _run_env(completion_text, st["current_prompt"])
            pred_smiles = extract_smiles(completion_text)
            valid_output_canonical = Chem.MolToSmiles(mol_out) if mol_out is not None else None

            st["turns"].append(
                {
                    "prompt_tokens": prompt_tokens[k],
                    "comp_tokens": list(seq.tokens),
                    "comp_logprobs": list(seq.logprobs),
                    "turn_reward": turn_reward,
                    "reward_components": reward_components,
                    "turn_valid_canonical_smiles": valid_output_canonical,
                }
            )

            if debug:
                print(
                    f"[DBG]{debug_prefix} traj={k} turn={turn_idx} "
                    f"reward={turn_reward:.4f} all_met={all_met} "
                    f"fmt={reward_components['strict']:.3f} "
                    f"prop={reward_components['property']:.3f} "
                    f"rep={reward_components['repetition']:.3f} "
                    f"valid_mol={mol_out is not None}"
                )
                print(f"[DBG]{debug_prefix} traj={k} turn={turn_idx} raw_output: {raw_completion_text!r}")
                print(f"[DBG]{debug_prefix} traj={k} turn={turn_idx} normalized_output: {completion_text}")

            turn_mol_in = _mol_from_prompt(st["current_prompt"])
            if pred_smiles and turn_mol_in is not None and mol_out is not None:
                unmet_props, progress_lines = _target_progress_lines(
                    turn_mol_in, mol_out, st["current_prompt"]
                )
                unmet_text = ", ".join(unmet_props) if unmet_props else "none"
                progress_block = (
                    "\n".join([f"- {line}" for line in progress_lines])
                    if progress_lines
                    else "- target-gap details unavailable."
                )
                st["prev_guidance"] = (
                    f"all_met={all_met}; unmet_properties={unmet_text}.\n{progress_block}"
                )
                st["current_valid_smiles"] = pred_smiles
                st["current_valid_canonical"] = valid_output_canonical
                st["has_valid_generated"] = True
                st["last_valid_generated_smiles"] = pred_smiles
                st["last_valid_generated_canonical"] = valid_output_canonical
                st["current_prompt"] = _replace_prompt_smiles(st["current_prompt"], pred_smiles)
                st["prev_env_feedback"] = _compact_env_feedback(env_feedback)
            else:
                rollback_feedback, rollback_guidance, _rollback_all_met = _rollback_state_feedback(
                    st["current_prompt"]
                )
                st["prev_guidance"] = rollback_guidance
                st["prev_env_feedback"] = _compact_env_feedback(rollback_feedback)
                if debug:
                    print(f"[DBG]{debug_prefix} traj={k} turn={turn_idx} rollback_state_feedback:")
                    print(rollback_feedback)

            if (turn_idx == max_turns - 1) or all_met:
                st["done"] = True

    def _final_result_canonical(st: dict) -> str | None:
        # Terminal invalid turns do not invalidate the trajectory; use the current rollback state.
        # If the policy never produced a valid molecule, keep final result as missing.
        if st.get("has_valid_generated"):
            return st.get("current_valid_canonical") or st.get("last_valid_generated_canonical")
        return None

    final_canonical_by_traj = [_final_result_canonical(st) for st in traj_states]

    # Prompt-local repetition penalty: only compare final valid molecules among this prompt's K trajectories.
    final_canonicals = [can for can in final_canonical_by_traj if can]
    final_counts = Counter(final_canonicals)
    for st, final_can in zip(traj_states, final_canonical_by_traj):
        repetition_count = max(int(final_counts.get(final_can, 0)) - 1, 0) if final_can else 0
        repetition_penalty = -REPETITION_PENALTY_STEP * repetition_count
        st["final_repetition_count"] = repetition_count
        st["final_repetition_penalty"] = float(repetition_penalty)
        if st["turns"]:
            last_turn = st["turns"][-1]
            comps = last_turn["reward_components"]
            comps["repetition"] = float(repetition_penalty)
            comps["repetition_count"] = int(repetition_count)
            comps["total"] = float(comps["total"] + repetition_penalty)
            last_turn["turn_reward"] = float(last_turn["turn_reward"] + repetition_penalty)

    final_nte_records = [
        compute_final_nte_details(final_can, prompt_text)
        for final_can in final_canonical_by_traj
    ]

    all_trajectories = []
    for st in traj_states:
        turns = st["turns"]
        n = len(turns)
        returns = [0.0] * n
        running = 0.0
        for i in reversed(range(n)):
            running = turns[i]["turn_reward"] + gamma * running
            returns[i] = running
        G_k = returns[0] if returns else 0.0
        all_trajectories.append((turns, returns, G_k))

        if debug:
            turn_rewards_dbg = [round(t["turn_reward"], 4) for t in turns]
            returns_dbg = [round(x, 4) for x in returns]
            traj_idx = traj_states.index(st)
            final_can_dbg = final_canonical_by_traj[traj_idx]
            print(
                f"[DBG]{debug_prefix} traj={traj_idx} "
                f"turn_rewards={turn_rewards_dbg} returns={returns_dbg} G_k={G_k:.4f} "
                f"final={final_can_dbg} "
                f"rep_count={st.get('final_repetition_count', 0)}"
            )

    G_values = [t[2] for t in all_trajectories]
    G_arr = np.array(G_values, dtype=np.float64)
    G_mean = G_arr.mean() if len(G_arr) else 0.0
    G_std = G_arr.std() if len(G_arr) else 0.0

    if G_std < 1e-8:
        relative_advs = [G - ema_baseline.value for G in G_values]
        for G in G_values:
            ema_baseline.update_and_get(G)
    else:
        relative_advs = ((G_arr - G_mean) / G_std).tolist()
        for G in G_values:
            ema_baseline.update_and_get(G)

    datums = []
    traj_returns = []
    dropped_zero_adv = 0
    for idx, ((turns, returns, G_k), adv) in enumerate(zip(all_trajectories, relative_advs)):
        traj_returns.append(float(G_k))
        if abs(adv) < 1e-12:
            dropped_zero_adv += 1
            if debug:
                print(f"[DBG]{debug_prefix} drop traj={idx} because adv≈0 ({adv:.4e})")
            continue
        for turn in turns:
            n_p = len(turn["prompt_tokens"])
            n_c = len(turn["comp_tokens"])
            full = turn["prompt_tokens"] + turn["comp_tokens"]
            min_len = min(
                len(full) - 1,
                n_p - 1 + n_c,
                n_p - 1 + len(turn["comp_logprobs"]),
            )
            datums.append(
                types.Datum(
                    model_input=types.ModelInput.from_ints(tokens=full[:-1]),
                    loss_fn_inputs=dict(
                        target_tokens=full[1:][:min_len],
                        advantages=([0.0] * (n_p - 1) + [adv] * n_c)[:min_len],
                        logprobs=([0.0] * (n_p - 1) + turn["comp_logprobs"])[:min_len],
                    ),
                )
            )

    adv_arr = np.array(relative_advs, dtype=np.float64) if relative_advs else np.array([], dtype=np.float64)
    turn_components = [
        turn["reward_components"]
        for traj_turns, _, _ in all_trajectories
        for turn in traj_turns
        if "reward_components" in turn
    ]

    def _mean_component(name: str) -> float:
        if not turn_components:
            return 0.0
        return float(np.mean([c.get(name, 0.0) for c in turn_components]))

    valid_nte_records = [r for r in final_nte_records if r.get("valid")]

    def _values(name: str) -> list[float]:
        return [float(r[name]) for r in valid_nte_records if r.get(name) is not None]

    nte_values = _values("nte")
    final_qed_errors = _values("qed_error")
    final_logp_errors = _values("logp_error")
    final_mw_errors = _values("mw_error")

    def _mean_or_none(values: list[float]):
        return float(np.mean(values)) if values else None

    def _std_or_none(values: list[float]):
        return float(np.std(values)) if values else None

    details = {
        "mean_G": float(G_mean),
        "std_G": float(G_std),
        "mean_adv": float(np.mean(adv_arr)) if adv_arr.size else 0.0,
        "mean_abs_adv": float(np.mean(np.abs(adv_arr))) if adv_arr.size else 0.0,
        "std_adv": float(np.std(adv_arr)) if adv_arr.size else 0.0,
        "pos_adv_ratio": float(np.mean(adv_arr > 0.0)) if adv_arr.size else 0.0,
        "num_traj": len(all_trajectories),
        "num_turns_total": sum(len(t[0]) for t in all_trajectories),
        "dropped_zero_adv": dropped_zero_adv,
        "mean_reward_total": _mean_component("total"),
        "mean_reward_format": _mean_component("strict"),
        "mean_reward_property": _mean_component("property"),
        "mean_reward_repetition": _mean_component("repetition"),
        "mean_repetition_count": _mean_component("repetition_count"),
        "num_final_valid": len(valid_nte_records),
        "num_final_missing": len(final_nte_records) - len(valid_nte_records),
        "mean_nte": _mean_or_none(nte_values),
        "std_nte": _std_or_none(nte_values),
        "mean_final_qed_error": _mean_or_none(final_qed_errors),
        "mean_final_logp_error": _mean_or_none(final_logp_errors),
        "mean_final_mw_error": _mean_or_none(final_mw_errors),
        "nte_values": nte_values,
        "final_qed_errors": final_qed_errors,
        "final_logp_errors": final_logp_errors,
        "final_mw_errors": final_mw_errors,
        "final_canonical_smiles": final_canonicals,
        "final_canonical_smiles_per_trajectory": final_canonical_by_traj,
    }
    return datums, traj_returns, details


In [19]:
# Create a NEW training client for GRPO, starting from the SFT checkpoint
grpo_training_client = service_client.create_lora_training_client(
    base_model=BASE_MODEL,
    rank=LORA_RANK,
)

# # Load the SFT weights as our starting point
grpo_training_client.load_state("tinker://da3b0ce4-fbcc-5619-8304-7752622fdc4e:train:2/weights/general-sft-1000")
print("Loaded SFT checkpoint into GRPO training client")

# Create initial sampling client
grpo_sampling_client = grpo_training_client.save_weights_and_get_sampling_client(
    name="grpo-init"
)
print("GRPO sampling client ready")

Loaded SFT checkpoint into GRPO training client


KeyboardInterrupt: 

In [53]:
from rdkit import Chem
from rdkit.Chem import BRICS

def get_clean_fragments(smiles: str):
    """
    Break a molecule into BRICS fragments and remove all attachment point markers (e.g., [4*]).

    Args:
        smiles (str): SMILES string of the input molecule.

    Returns:
        List[str]: List of cleaned fragment SMILES strings.
    """
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return []

    try:
        mol_broken = BRICS.BreakBRICSBonds(mol)
        fragments = Chem.GetMolFrags(mol_broken, asMols=True, sanitizeFrags=True)
        clean_frags = []
        for frag in fragments:
            # Remove attachment point atoms like [4*]
            editable = Chem.EditableMol(frag)
            to_remove = [atom.GetIdx() for atom in frag.GetAtoms() if atom.GetSymbol() == '*' and atom.GetAtomicNum() == 0]
            for idx in sorted(to_remove, reverse=True):
                editable.RemoveAtom(idx)
            cleaned = editable.GetMol()
            Chem.SanitizeMol(cleaned)
            clean_frags.append(Chem.MolToSmiles(cleaned, isomericSmiles=True))
        return clean_frags
    except Exception as e:
        print(f"Error in fragmenting molecule: {e}")
        return []

In [54]:
# Unified single-stage multi-turn GRPO training loop.
def run_multistep_rl_training_fixed(
    raw_data,
    renderer,
    training_client,
    tokenizer,
    num_iterations: int = 50,
    prompts_per_iter: int = 4,
    learning_rate: float = 3e-6,
    trajectories_per_prompt: int = 8,
    max_turns: int = 3,
    gamma: float = 0.85,
    ema_alpha: float = 0.05,
    rng_seed: int = 42,
    debug_adv: bool = False,
    debug_prompts_per_iter: int = 0,
    debug_iterations: int = 0,
    keep_skipped_metrics: bool = True,
    refresh_sampling_every: int = 6,
    temperature: float = 1.0,
    top_p: float = 0.9,
    top_k: int = 50,
):
    """Single-stage multi-turn GRPO with prompt-local repetition penalty."""
    if refresh_sampling_every < 1:
        raise ValueError("refresh_sampling_every must be >= 1")

    rng = np.random.default_rng(rng_seed)
    ema = EMABaseline(alpha=ema_alpha)
    sampling_client = training_client.save_weights_and_get_sampling_client(
        name="single-stage-init"
    )

    globals()["SAMPLING_TEMPERATURE_ACTIVE"] = float(temperature)
    globals()["SAMPLING_TOP_P_ACTIVE"] = float(top_p)
    globals()["SAMPLING_TOP_K_ACTIVE"] = int(top_k)
    globals()["TRAJECTORIES_PER_PROMPT_ACTIVE"] = int(trajectories_per_prompt)
    globals()["REFRESH_SAMPLING_EVERY_ACTIVE"] = int(refresh_sampling_every)

    all_metrics = []
    print("Starting single-stage multi-turn GRPO training")
    print("reward = format_r + prop_r + prompt-local repetition_penalty")
    print(
        f"sampling temperature={temperature:.2f}, top_p={top_p:.2f}, top_k={top_k}, "
        f"trajectories_per_prompt={trajectories_per_prompt}, refresh_sampling_every={refresh_sampling_every}"
    )
    header = f"{'Iter':>4} | {'Mean G':>8} | {'std_G':>7} | {'|adv|':>7} | {'std_adv':>7} | {'+adv%':>6} | {'#data':>6} | {'t(s)':>6}"
    print(header)
    print("-" * len(header))

    def _flatten_numeric(details_list, key: str) -> list[float]:
        values = []
        for d in details_list:
            for value in d.get(key, []) or []:
                if isinstance(value, (int, float)):
                    values.append(float(value))
        return values

    def _mean_or_none(values: list[float]):
        return float(np.mean(values)) if values else None

    def _std_or_none(values: list[float]):
        return float(np.std(values)) if values else None

    def _fmt_optional(value, digits: int = 4) -> str:
        return "nan" if value is None else f"{value:.{digits}f}"

    for iteration in range(num_iterations):
        import time
        t0 = time.time()

        prompt_indices = rng.choice(len(raw_data), size=prompts_per_iter, replace=False)
        all_datums = []
        all_rewards = []
        all_details = []

        for local_idx, prompt_idx in enumerate(prompt_indices):
            example = raw_data[int(prompt_idx)]
            try:
                do_debug = (
                    debug_adv
                    and iteration < debug_iterations
                    and local_idx < debug_prompts_per_iter
                )
                datums, rewards, details = run_mode_b(
                    example,
                    renderer,
                    sampling_client,
                    tokenizer,
                    ema_baseline=ema,
                    num_trajectories=trajectories_per_prompt,
                    max_turns=max_turns,
                    gamma=gamma,
                    temperature=temperature,
                    top_p=top_p,
                    top_k=top_k,
                    debug=do_debug,
                    debug_prefix=f" it={iteration} p={int(prompt_idx)} ",
                )
            except Exception as e:
                print(f"  [skip] prompt {prompt_idx}: {e}")
                continue

            all_datums.extend(datums)
            all_rewards.extend(rewards)
            all_details.append(details)

        elapsed = time.time() - t0

        nte_values = _flatten_numeric(all_details, "nte_values")
        final_qed_errors = _flatten_numeric(all_details, "final_qed_errors")
        final_logp_errors = _flatten_numeric(all_details, "final_logp_errors")
        final_mw_errors = _flatten_numeric(all_details, "final_mw_errors")
        num_final_valid = int(np.sum([d.get("num_final_valid", 0) for d in all_details]))
        num_final_missing = int(np.sum([d.get("num_final_missing", 0) for d in all_details]))
        mean_nte = _mean_or_none(nte_values)
        std_nte = _std_or_none(nte_values)
        mean_final_qed_error = _mean_or_none(final_qed_errors)
        mean_final_logp_error = _mean_or_none(final_logp_errors)
        mean_final_mw_error = _mean_or_none(final_mw_errors)

        if not all_datums:
            if all_details:
                mean_G = float(np.mean([d.get("mean_G", 0.0) for d in all_details]))
                std_G = float(np.mean([d.get("std_G", 0.0) for d in all_details]))
                mean_adv = float(np.mean([d.get("mean_adv", 0.0) for d in all_details]))
                mean_abs_adv = float(np.mean([d.get("mean_abs_adv", 0.0) for d in all_details]))
                std_adv = float(np.mean([d.get("std_adv", 0.0) for d in all_details]))
                pos_adv_ratio = float(np.mean([d.get("pos_adv_ratio", 0.0) for d in all_details]))
                dropped = int(np.sum([d.get("dropped_zero_adv", 0) for d in all_details]))
                mean_reward_total = float(np.mean([d.get("mean_reward_total", 0.0) for d in all_details]))
                mean_reward_format = float(np.mean([d.get("mean_reward_format", 0.0) for d in all_details]))
                mean_reward_property = float(np.mean([d.get("mean_reward_property", 0.0) for d in all_details]))
                mean_reward_repetition = float(np.mean([d.get("mean_reward_repetition", 0.0) for d in all_details]))

                print(
                    f"{iteration:>4} | no valid data, skipped "
                    f"(mean_G={mean_G:.4f}, std_G={std_G:.4f}, |adv|={mean_abs_adv:.4f}, "
                    f"std_adv={std_adv:.4f}, +adv%={100.0 * pos_adv_ratio:.1f}, dropped={dropped})"
                )
                print(
                    "      rewards | "
                    f"total={mean_reward_total:+.3f} | "
                    f"format={mean_reward_format:+.3f} | "
                    f"property={mean_reward_property:+.3f} | "
                    f"repetition={mean_reward_repetition:+.3f}"
                )
                print(
                    "      nte | "
                    f"mean={_fmt_optional(mean_nte)} std={_fmt_optional(std_nte)} "
                    f"valid={num_final_valid} missing={num_final_missing} | "
                    f"|dQED|={_fmt_optional(mean_final_qed_error)} | "
                    f"|dLogP|={_fmt_optional(mean_final_logp_error)} | "
                    f"|dMW|={_fmt_optional(mean_final_mw_error, digits=2)}"
                )

                if keep_skipped_metrics:
                    all_metrics.append(
                        {
                            "iteration": iteration,
                            "mean_G": mean_G,
                            "std_G": std_G,
                            "mean_adv": mean_adv,
                            "mean_abs_adv": mean_abs_adv,
                            "std_adv": std_adv,
                            "pos_adv_ratio": pos_adv_ratio,
                            "num_data": 0,
                            "skipped": True,
                            "dropped_zero_adv": dropped,
                            "mean_reward_total": mean_reward_total,
                            "mean_reward_format": mean_reward_format,
                            "mean_reward_property": mean_reward_property,
                            "mean_reward_repetition": mean_reward_repetition,
                            "mean_nte": mean_nte,
                            "std_nte": std_nte,
                            "num_final_valid": num_final_valid,
                            "num_final_missing": num_final_missing,
                            "mean_final_qed_error": mean_final_qed_error,
                            "mean_final_logp_error": mean_final_logp_error,
                            "mean_final_mw_error": mean_final_mw_error,
                            "sampling_temperature": float(temperature),
                            "sampling_top_p": float(top_p),
                            "sampling_top_k": int(top_k),
                            "trajectories_per_prompt": int(trajectories_per_prompt),
                            "refresh_sampling_every": int(refresh_sampling_every),
                            "time": elapsed,
                        }
                    )
            else:
                print(f"{iteration:>4} | no valid data, skipped")
                if keep_skipped_metrics:
                    all_metrics.append(
                        {
                            "iteration": iteration,
                            "num_data": 0,
                            "skipped": True,
                            "sampling_temperature": float(temperature),
                            "sampling_top_p": float(top_p),
                            "sampling_top_k": int(top_k),
                            "trajectories_per_prompt": int(trajectories_per_prompt),
                            "refresh_sampling_every": int(refresh_sampling_every),
                            "time": elapsed,
                        }
                    )
            continue

        fwdbwd = training_client.forward_backward(
            all_datums,
            loss_fn="ppo",
            loss_fn_config={"clip_low_threshold": 0.9, "clip_high_threshold": 1.1},
        )
        optim = training_client.optim_step(types.AdamParams(learning_rate=learning_rate))
        fwdbwd.result()
        optim.result()

        should_refresh = ((iteration + 1) % refresh_sampling_every == 0) or (iteration == num_iterations - 1)
        if should_refresh:
            sampling_client = training_client.save_weights_and_get_sampling_client(
                name=f"single-stage-iter-{iteration:04d}"
            )

        mean_G = float(np.mean([d.get("mean_G", 0.0) for d in all_details]))
        std_G = float(np.mean([d.get("std_G", 0.0) for d in all_details]))
        mean_adv = float(np.mean([d.get("mean_adv", 0.0) for d in all_details]))
        mean_abs_adv = float(np.mean([d.get("mean_abs_adv", 0.0) for d in all_details]))
        std_adv = float(np.mean([d.get("std_adv", 0.0) for d in all_details]))
        pos_adv_ratio = float(np.mean([d.get("pos_adv_ratio", 0.0) for d in all_details]))
        dropped = int(np.sum([d.get("dropped_zero_adv", 0) for d in all_details]))
        mean_reward_total = float(np.mean([d.get("mean_reward_total", 0.0) for d in all_details]))
        mean_reward_format = float(np.mean([d.get("mean_reward_format", 0.0) for d in all_details]))
        mean_reward_property = float(np.mean([d.get("mean_reward_property", 0.0) for d in all_details]))
        mean_reward_repetition = float(np.mean([d.get("mean_reward_repetition", 0.0) for d in all_details]))

        print(
            f"{iteration:>4} | {mean_G:>8.4f} | {std_G:>7.4f} | {mean_abs_adv:>7.4f} | "
            f"{std_adv:>7.4f} | {100.0 * pos_adv_ratio:>5.1f}% | {len(all_datums):>6} | {elapsed:>6.1f}"
        )
        print(
            "      rewards | "
            f"total={mean_reward_total:+.3f} | "
            f"format={mean_reward_format:+.3f} | "
            f"property={mean_reward_property:+.3f} | "
            f"repetition={mean_reward_repetition:+.3f}"
        )
        print(
            "      nte | "
            f"mean={_fmt_optional(mean_nte)} std={_fmt_optional(std_nte)} "
            f"valid={num_final_valid} missing={num_final_missing} | "
            f"|dQED|={_fmt_optional(mean_final_qed_error)} | "
            f"|dLogP|={_fmt_optional(mean_final_logp_error)} | "
            f"|dMW|={_fmt_optional(mean_final_mw_error, digits=2)}"
        )

        metric = dict(
            iteration=iteration,
            mean_G=mean_G,
            std_G=std_G,
            mean_adv=mean_adv,
            mean_abs_adv=mean_abs_adv,
            std_adv=std_adv,
            pos_adv_ratio=pos_adv_ratio,
            num_data=len(all_datums),
            dropped_zero_adv=dropped,
            mean_reward_total=mean_reward_total,
            mean_reward_format=mean_reward_format,
            mean_reward_property=mean_reward_property,
            mean_reward_repetition=mean_reward_repetition,
            mean_nte=mean_nte,
            std_nte=std_nte,
            num_final_valid=num_final_valid,
            num_final_missing=num_final_missing,
            mean_final_qed_error=mean_final_qed_error,
            mean_final_logp_error=mean_final_logp_error,
            mean_final_mw_error=mean_final_mw_error,
            sampling_temperature=float(temperature),
            sampling_top_p=float(top_p),
            sampling_top_k=int(top_k),
            trajectories_per_prompt=int(trajectories_per_prompt),
            refresh_sampling_every=int(refresh_sampling_every),
            ema_baseline=ema.value,
            time=elapsed,
            skipped=False,
            refreshed_sampling_client=bool(should_refresh),
        )
        all_metrics.append(metric)

    print(f"\nTraining complete ({num_iterations} iterations)")
    globals()["all_metrics"] = all_metrics
    return all_metrics


In [57]:
# Single-stage main GRPO experiment config.
METHOD_VARIANT = "nonsubprocess_single_stage"
RUN_EXPERIMENT = METHOD_VARIANT

cfg = dict(
    num_iterations=20,
    prompts_per_iter=2,
    learning_rate=2e-6,
    trajectories_per_prompt=8,
    max_turns=3,
    gamma=0.85,
    ema_alpha=0.05,
    rng_seed=42,
    debug_adv=False,
    debug_prompts_per_iter=0,
    debug_iterations=0,
    keep_skipped_metrics=True,
    refresh_sampling_every=4,
    temperature=1.0,
    top_p=0.9,
    top_k=50,
)

RUN_TAG = (
    f"{RUN_EXPERIMENT}_iters{cfg['num_iterations']}_"
    f"prompts{len(raw_data)}_samples{cfg['trajectories_per_prompt']}_"
    f"temp{cfg['temperature']}_top_p{cfg['top_p']}_top_k{cfg['top_k']}"
)

globals()["METHOD_VARIANT"] = METHOD_VARIANT
globals()["RUN_EXPERIMENT"] = RUN_EXPERIMENT
globals()["RUN_TAG"] = RUN_TAG

print(f"Running method={METHOD_VARIANT}")
print("Single-stage GRPO config:")
print(cfg)

all_metrics = run_multistep_rl_training_fixed(
    raw_data=raw_data,
    renderer=renderer,
    training_client=grpo_training_client,
    tokenizer=tokenizer,
    **cfg,
)

print(f"Training finished. Experiment={RUN_EXPERIMENT}, metric points={len(all_metrics)}")


Running method=nonsubprocess_single_stage
Single-stage GRPO config:
{'num_iterations': 20, 'prompts_per_iter': 2, 'learning_rate': 2e-06, 'trajectories_per_prompt': 8, 'max_turns': 3, 'gamma': 0.85, 'ema_alpha': 0.05, 'rng_seed': 42, 'debug_adv': False, 'debug_prompts_per_iter': 0, 'debug_iterations': 0, 'keep_skipped_metrics': True, 'refresh_sampling_every': 4, 'temperature': 1.0, 'top_p': 0.9, 'top_k': 50}


KeyboardInterrupt: 

In [ ]:
# Save the final GRPO checkpoint.
grpo_save_result = grpo_training_client.save_state(
    name=""
).result()
grpo_checkpoint_path = grpo_save_result.path
print(f"GRPO checkpoint saved at: {grpo_checkpoint_path}")


## 8. Plot Training Curves

In [ ]:
from datetime import datetime
from pathlib import Path
from matplotlib import pyplot as plt

if "all_metrics" not in globals():
    raise ValueError(
        "all_metrics is undefined. Run: all_metrics = run_multistep_rl_training_fixed(...) first."
    )
if not all_metrics:
    raise ValueError("all_metrics is empty. Please run training first.")


def _smooth_series(values, window: int = 3):
    arr = np.asarray([np.nan if v is None else v for v in values], dtype=np.float64)
    if arr.size == 0 or window <= 1 or arr.size < window:
        return arr.tolist()
    kernel = np.ones(window, dtype=np.float64) / window
    pad_left = window // 2
    pad_right = window - 1 - pad_left
    padded = np.pad(arr, (pad_left, pad_right), mode="edge")
    smooth = np.convolve(padded, kernel, mode="valid")
    return smooth.tolist()


iters = [m["iteration"] for m in all_metrics]
save_dir = Path("output/training_plots")
save_dir.mkdir(parents=True, exist_ok=True)


def _auto_run_tag() -> str:
    explicit = globals().get("RUN_TAG")
    if explicit:
        return str(explicit)
    parts = []
    experiment_name = globals().get("RUN_EXPERIMENT")
    if experiment_name:
        parts.append(str(experiment_name))
    parts.append(f"iters{len(all_metrics)}")
    if "raw_data" in globals():
        parts.append(f"prompts{len(raw_data)}")
    parts.append(datetime.now().strftime("%Y%m%d_%H%M%S_%f"))
    return "_".join(parts)


def _assert_fresh_output_path(path: Path, *, kind: str) -> None:
    if path.exists():
        raise FileExistsError(
            f"Refusing to overwrite existing {kind}: {path}. Change run_tag before saving."
        )


run_tag = _auto_run_tag()

fig, axes = plt.subplots(2, 2, figsize=(13, 8), sharex=True)
axes = axes.flatten()
main_metrics = [
    ("mean_G", "Mean Trajectory Return (G)", "Return"),
    ("std_G", "Std of Trajectory Return", "Std"),
    ("mean_abs_adv", "Mean |Advantage|", "|Adv|"),
    ("mean_nte", "Final Molecule NTE", "NTE"),
]
for ax, (key, title, ylabel) in zip(axes, main_metrics):
    values = [m.get(key) for m in all_metrics]
    ax.plot(iters, values, "o-", markersize=3, alpha=0.45, label="raw")
    ax.plot(iters, _smooth_series(values), linewidth=2.0, label="smooth")
    ax.set_title(title)
    ax.set_ylabel(ylabel)
    ax.set_xlabel("Iteration")
    ax.grid(True, alpha=0.3)
    ax.legend(loc="best", fontsize=8)
plt.suptitle("Single-stage Multi-turn GRPO Training Progress", fontsize=14)
plt.tight_layout()
main_plot_path = save_dir / f"training_progress_{run_tag}.png"
_assert_fresh_output_path(main_plot_path, kind="training plot")
fig.savefig(main_plot_path, dpi=200, bbox_inches="tight")
print(f"Saved main training plot to: {main_plot_path}")
plt.show()

fig2, axes2 = plt.subplots(2, 2, figsize=(13, 8), sharex=True)
axes2 = axes2.flatten()
reward_component_keys = [
    ("mean_reward_total", "Total Reward"),
    ("mean_reward_format", "Format Reward"),
    ("mean_reward_property", "Property Reward"),
    ("mean_reward_repetition", "Prompt-local Repetition Penalty"),
]
for ax, (metric_key, title) in zip(axes2, reward_component_keys):
    values = [m.get(metric_key, 0.0) for m in all_metrics]
    ax.plot(iters, values, color="#9ecae1", linewidth=1.0, alpha=0.35, label="raw")
    ax.plot(iters, _smooth_series(values), color="#08519c", linewidth=2.0, label="smooth")
    ax.set_title(title)
    ax.set_xlabel("Iteration")
    ax.set_ylabel("Reward")
    ax.grid(True, alpha=0.3)
    ax.legend(loc="best", fontsize=8)
plt.suptitle("Reward Decomposition by Iteration", fontsize=14)
plt.tight_layout()
reward_plot_path = save_dir / f"reward_decomposition_{run_tag}.png"
_assert_fresh_output_path(reward_plot_path, kind="reward decomposition plot")
fig2.savefig(reward_plot_path, dpi=200, bbox_inches="tight")
print(f"Saved reward decomposition plot to: {reward_plot_path}")
plt.show()

fig3, axes3 = plt.subplots(1, 3, figsize=(15, 4), sharex=True)
error_metrics = [
    ("mean_final_qed_error", "Final |dQED|"),
    ("mean_final_logp_error", "Final |dLogP|"),
    ("mean_final_mw_error", "Final |dMW|"),
]
for ax, (metric_key, title) in zip(axes3, error_metrics):
    values = [m.get(metric_key) for m in all_metrics]
    ax.plot(iters, values, "o-", markersize=3, alpha=0.45, label="raw")
    ax.plot(iters, _smooth_series(values), linewidth=2.0, label="smooth")
    ax.set_title(title)
    ax.set_xlabel("Iteration")
    ax.grid(True, alpha=0.3)
    ax.legend(loc="best", fontsize=8)
plt.suptitle("Final Molecule Property Errors", fontsize=14)
plt.tight_layout()
error_plot_path = save_dir / f"final_property_errors_{run_tag}.png"
_assert_fresh_output_path(error_plot_path, kind="final property error plot")
fig3.savefig(error_plot_path, dpi=200, bbox_inches="tight")
print(f"Saved final property error plot to: {error_plot_path}")
plt.show()

import importlib
import training_metrics_io as _training_metrics_io

_training_metrics_io = importlib.reload(_training_metrics_io)

metrics_artifacts = _training_metrics_io.save_training_metrics_from_globals(
    globals(),
    output_dir="out/training_metrics",
    run_name=f"multiturn_{run_tag}",
    allow_overwrite=False,
)
print(f"Saved metrics bundle to: {metrics_artifacts['run_dir']}")


## 9. Evaluate the Final Model

In [ ]:
sft_training_client = service_client.create_lora_training_client(
    base_model=BASE_MODEL,
    rank=LORA_RANK,
)
sft_training_client.load_state("tinker://74e5f0bc-93c9-5df1-9570-5ed0910d9daa:train:0/weights/sft-final")
sft_sampling_client = sft_training_client.save_weights_and_get_sampling_client(name="sft-sampler.weekend")
grpo_training_client = service_client.create_lora_training_client(
    base_model=BASE_MODEL,
    rank=LORA_RANK,
)
grpo_training_client.load_state("tinker://eb393fba-89e6-5005-b2f2-3df09d6142c3:train:0/weights/grpo-final.weekend")
grpo_sampling_client = grpo_training_client.save_weights_and_get_sampling_client(name="grpo-sampler.weekend")

In [ ]:
# Save the final GRPO checkpoint
grpo_save_result = grpo_training_client.save_state(name="MultistepRL-grpo-final.weekend.Qwen8B").result()
grpo_checkpoint_path = grpo_save_result.path
print(f"GRPO checkpoint saved at: {grpo_checkpoint_path}")

## 10. Download the Trained Weights

You can download the LoRA adapter weights to use with your own inference setup (e.g., vLLM, SGLang, or HuggingFace).

In [6]:
# Download the GRPO model weights
# The downloaded weights are a LoRA adapter that can be loaded
# with HuggingFace PEFT or merged into the base model.

# rest_client = service_client.create_rest_client()

# # List available checkpoints
# future = rest_client.list_checkpoints("run_id").result() 

rest_client = service_client.create_rest_client()
rest_client.delete_checkpoint(checkpoint_id="weights/2D-00050",training_run_id="8934373e-b2df-51b1-a352-ce60108949c1:train:0").result()  # Clean up any previous checkpoint to avoid confusion
 # Uncomment to see all checkpoints
# Download to a local directory
# Uncomment and run when ready:
# rest_client.download_checkpoint(grpo_checkpoint_path, output_dir="./grpo-weights")
# print("Weights downloaded to ./grpo-weights/")

# print(f"GRPO checkpoint path: {grpo_checkpoint_path}")
# print("Uncomment the download line above when you're ready to export.")

---

## Summary & Next Steps

In this notebook, we:

1. **Loaded** the `Alan123/homo-lumo-grpo` dataset from HuggingFace
2. **SFT-trained** a Qwen3 model to learn the chemistry Q&A format using `cross_entropy` loss
3. **GRPO-trained** the model using RL with a custom reward function and `ppo` loss
4. **Compared** SFT vs GRPO predictions side by side

### Tips for better results:

- **More SFT steps** — if your dataset is large, train for more steps or multiple epochs
- **Tune the reward function** — the reward function is the most important part of GRPO. Make it reflect what you actually care about.
- **Try different loss functions** — Tinker supports `importance_sampling`, `ppo`, `cispo`, and `dro`. See the [Loss Functions docs](https://tinker-docs.thinkingmachines.ai/losses)
- **Increase samples per prompt** — more completions per prompt = better advantage estimates
- **Use a larger model** — `Qwen/Qwen3-30B-A3B` (MoE) is very cost-effective and much more capable
- **Use the Tinker Cookbook** — the cookbook has optimized training loops with async operations, logging, and eval. See [rl/train.py](https://github.com/thinking-machines-lab/tinker-cookbook/blob/main/tinker_cookbook/rl/train.py)

### Key Tinker resources:
- [Tinker API Docs](https://tinker-docs.thinkingmachines.ai/)
- [Supervised Learning Guide](https://tinker-docs.thinkingmachines.ai/supervised-learning/sl-basic)
- [RL Training Guide](https://tinker-docs.thinkingmachines.ai/rl/rl-basic)
- [Model Lineup](https://tinker-docs.thinkingmachines.ai/model-lineup)
- [Tinker Cookbook (GitHub)](https://github.com/thinking-machines-lab/tinker-cookbook)

In [44]:
# 诊断（精简版）：查看指定 prompt 的逐轮原始输出、解析结果与环境反馈（无四步）
prompt_idx = 14584
max_turns = 3
DIAG_TEMPERATURE = 1.0
DIAG_TOP_P = 0.9
DIAG_TOP_K = 50
DIAG_MAX_TOKENS = 256

example = raw_data[int(prompt_idx)]
prompt_text = str(example[PROMPT_COL])
ground_truth = str(example[ANSWER_COL])

def _replace_prompt_smiles(base_prompt: str, current_smiles: str) -> str:
    """
    Replace current molecule and refresh per-property target deltas.

    - Refresh the fragments clause from the latest molecule for later turns.
    - Keep absolute target values fixed and recompute <QED>/<LogP>/<MW>
      deltas relative to the new current molecule.
    """
    new_prompt = re.sub(
        r"<SMILES>.*?</SMILES>",
        lambda _m: f"<SMILES>{current_smiles}</SMILES>",
        base_prompt,
        count=1,
        flags=re.DOTALL,
    )

    fragment_fn = globals().get("get_clean_fragments")
    if callable(fragment_fn):
        try:
            fragment_list = fragment_fn(current_smiles)
        except Exception:
            fragment_list = []
    else:
        fragment_list = []

    if not fragment_list:
        fragment_list = [current_smiles]

    fragment_repr = repr(fragment_list)
    new_prompt = re.sub(
        r"(<SMILES>.*?</SMILES>)(?:,\s*which is composed of fragments\s+.*?\.)?",
        lambda m: f"{m.group(1)}, which is composed of fragments {fragment_repr}.",
        new_prompt,
        count=1,
        flags=re.DOTALL,
    )

    mol_old = _mol_from_prompt(base_prompt)
    mol_new = Chem.MolFromSmiles(current_smiles)
    if mol_old is None or mol_new is None:
        return new_prompt

    prop_specs = [
        ("QED", QED.qed, True),
        ("LogP", Crippen.MolLogP, False),
        ("MW", Descriptors.ExactMolWt, False),
    ]

    for prop, fn, is_qed in prop_specs:
        m = re.search(rf"<{prop}>(-?\d*\.?\d+)</{prop}>\s*(higher|lower)", new_prompt)
        if not m:
            continue

        old_delta = float(m.group(1))
        old_dir = m.group(2).lower()
        old_base = float(fn(mol_old))
        target_abs = old_base + (old_delta if old_dir == "higher" else -old_delta)
        if is_qed:
            target_abs = max(0.01, min(0.99, target_abs))

        new_base = float(fn(mol_new))
        new_dir = "higher" if target_abs >= new_base else "lower"
        new_delta = abs(target_abs - new_base)

        new_prompt = re.sub(
            rf"<{prop}>-?\d*\.?\d+</{prop}>\s*(higher|lower)",
            f"<{prop}>{new_delta:.3f}</{prop}> {new_dir}",
            new_prompt,
            count=1,
        )

    return new_prompt

def _compact_env_feedback(feedback_text: str) -> str:
    """Keep only headline + final summary to avoid repeating target lines."""
    if not feedback_text:
        return ""
    lines = [ln.strip() for ln in feedback_text.splitlines() if ln.strip()]
    if not lines:
        return feedback_text
    if lines[0].startswith("Environment feedback:"):
        if len(lines) == 1:
            return lines[0]
        tail = lines[-1]
        if tail == lines[0]:
            return lines[0]
        return lines[0] + "\n" + tail
    return feedback_text

def _rollback_state_feedback(current_prompt: str) -> tuple[str, str, bool]:
    rollback_mol = _mol_from_prompt(current_prompt)
    if rollback_mol is None:
        feedback = "Environment feedback: rollback molecule is missing or invalid in the current prompt."
        guidance = (
            "all_met=False; unmet_properties=unknown.\n"
            "- target-gap details unavailable because the rollback molecule is invalid."
        )
        return feedback, guidance, False

    qed_val = QED.qed(rollback_mol)
    logp_val = Crippen.MolLogP(rollback_mol)
    mw_val = Descriptors.ExactMolWt(rollback_mol)
    misses, progress_lines = _target_progress_lines(rollback_mol, rollback_mol, current_prompt)
    all_met_state = bool(progress_lines) and len(misses) == 0
    unmet_text = ", ".join(misses) if misses else ("none" if progress_lines else "unknown")
    guidance_block = (
        "\n".join([f"- {line}" for line in progress_lines])
        if progress_lines
        else "- target-gap details unavailable."
    )
    progress_block = (
        "\n- " + "\n- ".join(progress_lines)
        if progress_lines
        else "\n- No parseable target tags (<QED>/<LogP>/<MW>) found in prompt."
    )
    if all_met_state:
        summary = "Rollback molecule already meets all targets within tolerance."
    else:
        summary = f"Rollback molecule targets not met: {unmet_text}. Continue refining from this valid molecule."
    feedback = (
        "Environment feedback: previous attempt invalid/missing; using previous valid molecule "
        f"with QED={qed_val:.3f}, LogP={logp_val:.2f}, MW={mw_val:.1f}."
        f"\nTarget progress:{progress_block}"
        f"\n{summary}"
    )
    guidance = f"all_met={all_met_state}; unmet_properties={unmet_text}.\n{guidance_block}"
    return feedback, guidance, all_met_state

mol_in = _mol_from_prompt(prompt_text)
if mol_in is not None:
    qed_val = QED.qed(mol_in)
    logp_val = Crippen.MolLogP(mol_in)
    mw_val = Descriptors.ExactMolWt(mol_in)
else:
    qed_val = logp_val = mw_val = float("nan")

print("=" * 100)
print(f"Prompt idx={prompt_idx}")
print(prompt_text)
print(ground_truth)
print(f"Input molecule: QED={qed_val:.4f} logP={logp_val:.4f} MW={mw_val:.2f}")
print(
    "Diagnostic sampling: "
    f"temperature={DIAG_TEMPERATURE}, top_p={DIAG_TOP_P}, top_k={DIAG_TOP_K}, max_tokens={DIAG_MAX_TOKENS}"
)
print("=" * 100)

current_prompt_for_generation = prompt_text
prev_env_feedback = ""
prev_guidance = ""
final_valid_smiles = None
final_valid_canonical = None
final_valid_source_turn = None

for turn_idx in range(max_turns):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT_RL},
        {"role": "user", "content": current_prompt_for_generation},
    ]
    if turn_idx > 0 and (prev_env_feedback or prev_guidance):
        feedback_parts = [
            "Environment feedback from your previous attempt:",
            prev_env_feedback,
            "",
            "Target-gap priorities from previous turn (must address first):",
            prev_guidance,
            "",
            "Interpretation rules (must follow):",
            "- If a property says 'need ... more increase/decrease', move in that direction.",
            "- If a property says 'overshot by ...', reverse slightly toward target.",
            "- Keep already-met properties near target while fixing unmet ones.",
            "",
            "Use the current molecule shown in the prompt above as the only edit base.",
            "Do not restart from the original molecule.",
        ]
        messages.append(
            {
                "role": "user",
                "content": "\n".join(feedback_parts).strip(),
            }
        )

    print(f"\n[TURN {turn_idx}] messages sent to model:")
    for msg_i, msg in enumerate(messages):
        print(f"--- message[{msg_i}] role={msg.get('role')} ---")
        print(msg.get("content", ""))

    prompt_input = renderer.build_generation_prompt(messages)
    result = grpo_sampling_client.sample(
        prompt=prompt_input,
        sampling_params=types.SamplingParams(
            max_tokens=DIAG_MAX_TOKENS,
            temperature=DIAG_TEMPERATURE,
            top_p=DIAG_TOP_P,
            top_k=DIAG_TOP_K,
            stop=renderer.get_stop_sequences(),
        ),
        num_samples=1,
    ).result()

    seq = result.sequences[0]
    raw_completion_text = tokenizer.decode(
        seq.tokens,
        skip_special_tokens=True,
    )
    completion_text = _normalize_completion_text(raw_completion_text)

    print(f"\n[TURN {turn_idx}] model output (raw):")
    print(raw_completion_text)
    print(f"[TURN {turn_idx}] model output (normalized):")
    print(completion_text)

    pred_smiles = extract_smiles(completion_text)
    reward_components = compute_reward_components(completion_text, current_prompt_for_generation, ground_truth)
    reward = float(reward_components["total"])
    env_feedback, mol_out, all_met = _run_env(completion_text, current_prompt_for_generation)

    turn_mol_in = _mol_from_prompt(current_prompt_for_generation)
    current_can = Chem.MolToSmiles(turn_mol_in) if turn_mol_in is not None else None
    output_can = Chem.MolToSmiles(mol_out) if mol_out is not None else None
    molecule_changed = (current_can is not None and output_can is not None and current_can != output_can)

    print(f"[TURN {turn_idx}] parsed_smiles={pred_smiles!r}")
    print(f"[TURN {turn_idx}] current_canonical={current_can!r}")
    print(f"[TURN {turn_idx}] output_canonical={output_can!r}")
    print(f"[TURN {turn_idx}] molecule_changed={molecule_changed} valid_mol={mol_out is not None} all_met={all_met}")
    print(
        f"[TURN {turn_idx}] rewards | total={reward:+.3f} | "
        f"format={reward_components['strict']:+.3f} | "
        f"property={reward_components['property']:+.3f} | "
        f"repetition={reward_components['repetition']:+.3f}"
    )
    print(f"[TURN {turn_idx}] environment feedback:")
    print(env_feedback)

    if pred_smiles and turn_mol_in is not None and mol_out is not None:
        unmet_props, progress_lines = _target_progress_lines(
            turn_mol_in,
            mol_out,
            current_prompt_for_generation,
        )
        unmet_text = ", ".join(unmet_props) if unmet_props else "none"
        if progress_lines:
            progress_block = "\n".join([f"- {line}" for line in progress_lines])
        else:
            progress_block = "- target-gap details unavailable."
        prev_guidance = (
            f"all_met={all_met}; unmet_properties={unmet_text}.\n"
            f"{progress_block}"
        )
        final_valid_smiles = pred_smiles
        final_valid_canonical = output_can
        final_valid_source_turn = turn_idx
        current_prompt_for_generation = _replace_prompt_smiles(current_prompt_for_generation, pred_smiles)
        prev_env_feedback = _compact_env_feedback(env_feedback)
        print(f"[TURN {turn_idx}] accepted output; next prompt uses output molecule.")
    else:
        rollback_feedback, prev_guidance, _rollback_all_met = _rollback_state_feedback(
            current_prompt_for_generation
        )
        prev_env_feedback = _compact_env_feedback(rollback_feedback)
        print(f"[TURN {turn_idx}] rollback: invalid/missing molecule; next prompt keeps previous valid molecule.")
        print(f"[TURN {turn_idx}] rollback state feedback for next turn:")
        print(rollback_feedback)
    is_last = (turn_idx == max_turns - 1) or all_met
    if is_last:
        break

print("\n" + "=" * 100)
if final_valid_canonical:
    final_mol = Chem.MolFromSmiles(final_valid_canonical)
    final_qed = QED.qed(final_mol)
    final_logp = Crippen.MolLogP(final_mol)
    final_mw = Descriptors.ExactMolWt(final_mol)
    final_nte = compute_final_nte_details(final_valid_canonical, prompt_text)
    print(
        f"Final selected molecule: last valid generated molecule from turn {final_valid_source_turn}; "
        "terminal invalid turns are treated as rollback, not trajectory failure."
    )
    print(f"Final SMILES: {final_valid_smiles}")
    print(f"Final canonical: {final_valid_canonical}")
    print(f"Final properties: QED={final_qed:.4f} LogP={final_logp:.4f} MW={final_mw:.2f}")
    if final_nte.get("valid"):
        print(
            f"Final NTE={final_nte['nte']:.4f} | "
            f"|dQED|={final_nte['qed_error']:.4f} | "
            f"|dLogP|={final_nte['logp_error']:.4f} | "
            f"|dMW|={final_nte['mw_error']:.2f}"
        )
else:
    print("Final selected molecule: missing because this trajectory did not generate any valid molecule.")
print("=" * 100)


Prompt idx=14584
Given the intermediate molecule SMILES <SMILES>CC[C@H]1CC[C@](O)([C@]2(C#N)CCc3ccccc32)C1</SMILES>, which is composed of fragments ['CC', '[C@H]1CC[C@](O)([C@]2(C#N)CCc3ccccc32)C1']. Propose a single replace, add or remove step on fragment level that makes the new molecule's QED <QED>0.079</QED> lower, LogP <LogP>0.350</LogP> lower, and Molecular Weight <MW>31.076</MW> higher.
Replace [C@H]1CC[C@](O)([C@]2(C#N)CCc3ccccc32)C1 with [C@H]1CCC(=O)[C@H]([C@@H]2CCC[C@H](S(C)(=O)=O)C2)C1 to form <SMILES>CC[C@H]1CCC(=O)[C@H]([C@@H]2CCC[C@H](S(C)(=O)=O)C2)C1</SMILES>.
Input molecule: QED=0.8805 logP=3.3354 MW=255.16
Diagnostic sampling: temperature=1.0, top_p=0.9, top_k=50, max_tokens=256

[TURN 0] messages sent to model:
--- message[0] role=system ---

You are a helpful chemistry assistant for multi-turn molecule optimization.

Rules:
1) Output exactly one edit action sentence in one of these formats:
   - Replace <old_fragment> with <new_fragment> to form <SMILES>new_smiles_h

In [ ]:
# Stage comparison utilities are intentionally disabled for the single-stage GRPO refactor.
# Use the main single-stage config cell above and the saved metrics bundle for analysis.
print("Stage comparison utilities disabled: this notebook now uses a single-stage GRPO protocol.")


In [ ]:
# Disabled: old stage2 A/B branch run removed by the single-stage GRPO refactor.
print("Old stage2 branch A run disabled. Use the single-stage config cell instead.")


In [ ]:
# Disabled: old stage2 A/B branch run removed by the single-stage GRPO refactor.
print("Old stage2 branch B run disabled. Use the single-stage config cell instead.")


# 诊断

In [17]:
import importlib
import reward_saturation_diagnostic as rsd

rsd = importlib.reload(rsd)

records, summary, artifacts = rsd.run_reward_saturation_diagnostic(
    globals(),
    checkpoint="800",          # 也可以改成 "700"
    num_prompts=32,
    samples_per_prompt=12,
    temperature=0.9,
    max_tokens=128,
)


[rewarddiag] prompt 1/32 idx=550 done
[rewarddiag] prompt 2/32 idx=749 done
[rewarddiag] prompt 3/32 idx=1950 done
[rewarddiag] prompt 4/32 idx=3770 done
[rewarddiag] prompt 5/32 idx=4002 done
[rewarddiag] prompt 6/32 idx=4213 done
[rewarddiag] prompt 7/32 idx=14681 done
[rewarddiag] prompt 8/32 idx=15469 done
[rewarddiag] prompt 9/32 idx=19260 done
[rewarddiag] prompt 10/32 idx=29811 done
[rewarddiag] prompt 11/32 idx=33582 done
[rewarddiag] prompt 12/32 idx=50638 done
[rewarddiag] prompt 13/32 idx=56841 done
[rewarddiag] prompt 14/32 idx=72337 done
[rewarddiag] prompt 15/32 idx=87754 done
[rewarddiag] prompt 16/32 idx=89858 done
[rewarddiag] prompt 17/32 idx=92517 done
[rewarddiag] prompt 18/32 idx=96868 done
[rewarddiag] prompt 19/32 idx=97365 done
[rewarddiag] prompt 20/32 idx=98719 done
[rewarddiag] prompt 21/32 idx=102897 done
[rewarddiag] prompt 22/32 idx=117802 done
[rewarddiag] prompt 23/32 idx=119472 done
[rewarddiag] prompt 24/32 idx=122706 done
[rewarddiag] prompt 25/32 idx